# 🔧 Yardımcı Fonksiyonlar (Helper Functions)

In [72]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, r2_score, f1_score
import xgboost as xgb

def train_and_evaluate_models(df, target_column):
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    models = {
        "Random Forest": RandomForestClassifier() if y.nunique() < 10 else RandomForestRegressor(),
        "Logistic Regression": LogisticRegression() if y.nunique() < 10 else None,
        "Artificial Neural Network": MLPClassifier() if y.nunique() < 10 else MLPRegressor(),
        "SVM": SVC() if y.nunique() < 10 else SVR(),
        "Gradient Boosting": GradientBoostingClassifier() if y.nunique() < 10 else GradientBoostingRegressor(),
        "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss') if y.nunique() < 10 else xgb.XGBRegressor(),
        "Decision Tree": DecisionTreeClassifier() if y.nunique() < 10 else DecisionTreeRegressor(),
        "KNN": KNeighborsClassifier() if y.nunique() < 10 else KNeighborsRegressor()
    }
    
    results = []
    
    for name, model in models.items():
        if model is None:
            continue
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_train_pred = model.predict(X_train)
        
        if y.nunique() < 10:
            train_score = accuracy_score(y_train, y_train_pred)
            test_score = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted')
        else:
            train_score = r2_score(y_train, y_train_pred)
            test_score = r2_score(y_test, y_pred)
            f1 = None  # F1 skoru sadece sınıflandırma için geçerlidir
        
        results.append({
            "Model": name,
            "Train Score": train_score,
            "Test Score": test_score,
            "F1 Score": f1
        })
    
    results_df = pd.DataFrame(results)
    print(results_df)
    
    return results_df


In [73]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

def encode_categorical_columns(df):
    le = LabelEncoder()
    categorical_cols = df.select_dtypes(include=['object']).columns  # Sayısal olmayan sütunları seç
    
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col])  # Label encoding işlemi
    
    return df